# Intelligent Health Facility Recommendation System: Foundation

This notebook prepares the data required for the recommendation engine by combining cleaned facility information with the healthcare services offered at each facility. The resulting master dataset will serve as the primary data source for building and evaluating the recommendation system.

By the end of this notebook, we will:

- Load the cleaned KMHFR datasets.
- Merge facility and service information into a master dataset.
- Validate the merged data.
- Save the master dataset for downstream tasks.
- Develop a minimal service taxonomy for common healthcare needs

## 1. Import Required Libraries

This project uses Python libraries for data manipulation, visualization, and file management. These libraries will support data loading, preprocessing, exploratory analysis, and later stages of the recommendation engine development.

In [ ]:
#Data manipulation
import pandas as pd
import numpy as np

#Google Drive access
from google.colab import drive

#Visualisation
import matplotlib.pyplot as plt

# Configure pandas display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

print("Libraries imported successfully.")



Libraries imported successfully.


## 2. Mount Google Drive

The cleaned KMHFR datasets are stored in Google Drive. Mounting Google Drive allows the notebook to securely access these files throughout the project without repeatedly uploading them.

Using a fixed project directory also improves the reproducibility and organization of the notebook.

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Define project file

# Project folder
PROJECT_FOLDER = "/content/drive/MyDrive/Data/Afya guide/Clean_Data"

# Dataset paths
FACILITIES_PATH = f"{PROJECT_FOLDER}/clean_health_facilities.csv"
SERVICES_PATH = f"{PROJECT_FOLDER}/kmhfr_services_cleaned.csv"

print("Facilities file:", FACILITIES_PATH)
print("Services file :", SERVICES_PATH)

Facilities file: /content/drive/MyDrive/Data/Afya guide/Clean_Data/clean_health_facilities.csv
Services file : /content/drive/MyDrive/Data/Afya guide/Clean_Data/kmhfr_services_cleaned.csv


In [ ]:
import os
# Check that the project folder exists
print("Project folder exists:", os.path.exists(PROJECT_FOLDER))

# List all files in the project folder
print("\nFiles in project folder:")
for file in os.listdir(PROJECT_FOLDER):
    print("-", file)

Project folder exists: True

Files in project folder:
- clean_health_facilities.csv
- kmhfr_services_cleaned.csv
- service_mapping.csv
- master_facility_list.csv
- master_facility_list_cleaned.csv


## 4. Load the Cleaned Datasets

The project uses two cleaned datasets:

- **Health Facilities Dataset** – contains information about registered health facilities, including their location, ownership, level of care, operational status, and geographic coordinates.
- **Health Services Dataset** – contains the healthcare services offered by each facility.

After loading the datasets, we perform a preliminary inspection to verify that the files have been imported correctly and to understand their dimensions and structure before merging.

In [ ]:
#Load the cleaned datasets
facilities=pd.read_csv(FACILITIES_PATH)
services=pd.read_csv(SERVICES_PATH)

#Display dataset dimensions
print("="*60)
print("DATASET LOADED SUCCESSFULLY")
print("="*60)
print("Facilities dataset shape:",facilities.shape)
print("Services dataset shape:",services.shape)
#

DATASET LOADED SUCCESSFULLY
Facilities dataset shape: (12050, 40)
Services dataset shape: (205078, 9)


## 5. Inspect the Dataset Structure

Before combining the datasets, we examine their structure to identify the common identifier that links health facilities to the services they provide. Understanding the available variables and their data types helps ensure a correct and reliable merge.

This step also confirms that the datasets contain the necessary information required for the recommendation engine.

In [ ]:
def inspect_dataset(df, dataset_name):
    """
    Display a quick overview of a dataset.
    """

    print("=" * 80)
    print(f"{dataset_name}")
    print("=" * 80)

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    print("\nColumn Names")
    print("-" * 80)
    print(df.columns.tolist())

    print("\nData Types")
    print("-" * 80)
    display(df.dtypes.to_frame(name="Data Type"))

    print("\nMissing Values")
    print("-" * 80)
    display(df.isnull().sum().to_frame(name="Missing Values"))

    print("\nFirst Five Rows")
    print("-" * 80)
    display(df.head())

In [ ]:
inspect_dataset(facilities, "Facilities Dataset")

Facilities Dataset
Shape: 12,050 rows × 40 columns

Column Names
--------------------------------------------------------------------------------
['Facility ID', 'Facility Name', 'Facility URL', 'Facility Type', 'Keph Level', 'KHIS reporting', 'Open 24 hours', 'Published', 'Date established', 'Date requested', 'Date approved', 'Regulated', 'Regulation status', 'Regulating body', 'Registration number', 'License number', 'Category', 'Owner', 'County', 'Sub County', 'Ward', 'Latitude', 'Longitude', 'Town', 'Description', 'Nearest landmark', 'Plot number', 'Total In-patient beds', 'General In-patient beds', 'Cots', 'Maternity beds', 'Emergency casualty beds', 'Intensive Care Unit beds', 'High Dependency Unit beds', 'Isolation beds', 'General theatres', 'Maternity theatres', 'Open weekends', 'NHIF accreditation', 'Open late night']

Data Types
--------------------------------------------------------------------------------


,Data Type
Facility ID,object
Facility Name,object
Facility URL,object
Facility Type,object
Keph Level,object
KHIS reporting,object
Open 24 hours,object
Published,object
Date established,object
Date requested,object



Missing Values
--------------------------------------------------------------------------------


,Missing Values
Facility ID,0
Facility Name,0
Facility URL,0
Facility Type,0
Keph Level,0
KHIS reporting,1607
Open 24 hours,343
Published,160
Date established,162
Date requested,50



First Five Rows
--------------------------------------------------------------------------------


,Facility ID,Facility Name,Facility URL,Facility Type,Keph Level,KHIS reporting,Open 24 hours,Published,Date established,Date requested,Date approved,Regulated,Regulation status,Regulating body,Registration number,License number,Category,Owner,County,Sub County,Ward,Latitude,Longitude,Town,Description,Nearest landmark,Plot number,Total In-patient beds,General In-patient beds,Cots,Maternity beds,Emergency casualty beds,Intensive Care Unit beds,High Dependency Unit beds,Isolation beds,General theatres,Maternity theatres,Open weekends,NHIF accreditation,Open late night
0,003564d4-7ad6-4a97-8323-8eda7b3475a8,Mata Arba Dispensary,https://kmhfr.health.go.ke/public/facilities/003564d4-7ad6-4a97-8323-8eda7b3475a8,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2014-10-06,2016-02-12,Yes,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Isiolo,Cherab,Cherab,1.093142,38.420842,-,-,-,-,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN
2,001ff20b-d75a-4377-a762-18b6d5345d30,Arito Langi Health Centre,https://kmhfr.health.go.ke/public/facilities/001ff20b-d75a-4377-a762-18b6d5345d30,Basic Health Centre,Level 3,Yes,Yes,Yes,2016-03-17,2009-10-31,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Kisumu,Seme,West Seme,-0.094480,34.449480,-,3 km from Kolenyo market along Kisumu Bondo road and is 11 kms from the district office,-,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
3,0012e77d-f49b-4197-b4bb-bed6f21d4933,Rhonda Health Centre,https://kmhfr.health.go.ke/public/facilities/0012e77d-f49b-4197-b4bb-bed6f21d4933,Comprehensive Health Centre,Level 3,Yes,Yes,Yes,2016-03-17,2014-01-30,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Nakuru,Nakuru West,Rhoda,-0.289467,36.034856,-,Near Soko mjinga,-,nil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
4,001e2b20-c18f-459b-b095-eb6e9d082db0,Maxcare Medical Clinic,https://kmhfr.health.go.ke/public/facilities/001e2b20-c18f-459b-b095-eb6e9d082db0,Medical Clinic,Level 2,Yes,Yes,Yes,2018-04-12,2019-09-24,2021-03-10,Yes,Licensed,-,-,-,Private Practice,Private Practice - Clinical Officer,Kiambu,Gatundu North,Githobokoni,-0.887340,36.804480,Gakoe Shopping Center,Near Gakoe Catholic Church,Gakoe catholic church,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN


In [ ]:
inspect_dataset(services, "Services Dataset")

Services Dataset
Shape: 205,078 rows × 9 columns

Column Names
--------------------------------------------------------------------------------
['Facility ID', 'Facility Name', 'Facility URL', 'Service', 'Category', 'Status', 'Original_Service', 'Standardized_Service', 'Service_Group']

Data Types
--------------------------------------------------------------------------------


,Data Type
Facility ID,object
Facility Name,object
Facility URL,object
Service,object
Category,object
Status,object
Original_Service,object
Standardized_Service,object
Service_Group,object



Missing Values
--------------------------------------------------------------------------------


,Missing Values
Facility ID,0
Facility Name,0
Facility URL,0
Service,0
Category,0
Status,0
Original_Service,0
Standardized_Service,0
Service_Group,0



First Five Rows
--------------------------------------------------------------------------------


,Facility ID,Facility Name,Facility URL,Service,Category,Status,Original_Service,Standardized_Service,Service_Group
0,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/00846514-0b95-4503-b882-a603153b86f5,General Outpatient,CURATIVE SERVICES,Active,General Outpatient,General Outpatient Services,General Outpatient Services
1,00846514-0b95-4503-b882-a603153b86f5,Miomponi,https://kmhfr.health.go.ke/public/facilities/00846514-0b95-4503-b882-a603153b86f5,Focused Antenatal Care,ANTENATAL CARE,Active,Focused Antenatal Care,Focused Antenatal Care,Maternal Health Services
2,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Basic IMCI-management of acute Infections,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active,Basic IMCI-management of acute Infections,Basic IMCI - Management of Acute Infections,Child Health Services
3,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Integrated Management of Newborn & Childhood Illnesses,INTEGRATED MANAGEMENT OF CHILDHOOD ILLNESS,Active,Integrated Management of Newborn & Childhood Illnesses,Integrated Management of Newborn and Childhood Illnesses,Child Health Services
4,00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Shinyalu Model Health Centre,https://kmhfr.health.go.ke/public/facilities/00d4fbf3-5089-4f79-ad59-8e4432a15a6f,Condom Distribution & STI Prevention,HIV/AIDS PREVENTION AND CARE SERVICES,Active,Condom Distribution & STI Prevention,Condom Distribution and STI Prevention,Family Planning Services


## Dataset Inspection Summary

The cleaned KMHFR datasets were successfully loaded and inspected before merging. The inspection confirmed that the datasets have the required structure to support the development of a health facility recommendation system.

### Facilities Dataset

The health facilities dataset contains **12,050 registered health facilities** with **40 variables**. Each record represents an individual health facility and includes key attributes required for recommendation, including:

- Facility identification details
- Facility type and level of care
- Ownership and category
- Geographic information (county, sub-county, ward, latitude, longitude)
- Operational characteristics
- Facility capacity indicators such as inpatient beds and theatres

The dataset contains minimal missing geographic information, with most facilities having complete location and identification details. Some missing values exist in optional operational fields such as NHIF accreditation, opening hours, and specialized capacity indicators.

### Services Dataset

The services dataset contains **205,078 service records** with **9 variables**. Unlike the facilities dataset, this dataset follows a one-to-many structure where a single facility can provide multiple healthcare services.

The dataset includes:

- Facility identifier
- Service name
- Service category
- Service status
- Standardized service name
- Service group classification

The presence of standardized service names provides a strong foundation for building the service taxonomy required by the recommendation engine.


Both datasets contain the common identifier **Facility ID**, which will be used as the primary key for merging. The facilities dataset represents the facility profile, while the services dataset provides the healthcare services offered at each facility.


## Dataset Inspection Summary

The cleaned KMHFR datasets were successfully loaded and inspected before merging. The inspection confirmed that the datasets have the required structure to support the development of a health facility recommendation system.

### Facilities Dataset

The health facilities dataset contains **12,050 registered health facilities** with **40 variables**. Each record represents an individual health facility and includes key attributes required for recommendation, including:

- Facility identification details
- Facility type and level of care
- Ownership and category
- Geographic information (county, sub-county, ward, latitude, longitude)
- Operational characteristics
- Facility capacity indicators such as inpatient beds and theatres

The dataset contains minimal missing geographic information, with most facilities having complete location and identification details. Some missing values exist in optional operational fields such as NHIF accreditation, opening hours, and specialized capacity indicators.

### Services Dataset

The services dataset contains **205,078 service records** with **9 variables**. Unlike the facilities dataset, this dataset follows a one-to-many structure where a single facility can provide multiple healthcare services.

The dataset includes:

- Facility identifier
- Service name
- Service category
- Service status
- Standardized service name
- Service group classification

The presence of standardized service names provides a strong foundation for building the service taxonomy required by the recommendation engine.

### Merge Readiness Assessment

Both datasets contain the common identifier **Facility ID**, which will be used as the primary key for merging. The facilities dataset represents the facility profile, while the services dataset provides the healthcare services offered at each facility.

Before creating the master dataset, the relationship between the two datasets will be validated by checking:

- Uniqueness of Facility IDs
- Duplicate identifiers
- Facilities without service records
- Service records without matching facilities

This validation ensures that the merged dataset maintains data integrity and accurately represents facility-service relationships.

In [ ]:
# Validate Facility ID

print("=" * 60)
print("MERGE KEY VALIDATION")
print("=" * 60)

print(f"Unique Facility IDs in facilities dataset : {facilities['Facility ID'].nunique():,}")
print(f"Unique Facility IDs in services dataset   : {services['Facility ID'].nunique():,}")

print("\nDuplicate Facility IDs")
print("-" * 60)

print(f"Facilities dataset duplicates: {facilities['Facility ID'].duplicated().sum():,}")
print(f"Services dataset duplicates  : {services['Facility ID'].duplicated().sum():,}")

MERGE KEY VALIDATION
Unique Facility IDs in facilities dataset : 12,050
Unique Facility IDs in services dataset   : 14,169

Duplicate Facility IDs
------------------------------------------------------------
Facilities dataset duplicates: 0
Services dataset duplicates  : 190,909


In [ ]:
# Check unmatched Facility IDs

facility_ids = set(facilities["Facility ID"])
service_ids = set(services["Facility ID"])

missing_services = facility_ids - service_ids
missing_facilities = service_ids - facility_ids

print("=" * 60)
print("MATCHING CHECK")
print("=" * 60)

print(f"Facilities without service records : {len(missing_services):,}")
print(f"Service records without facilities : {len(missing_facilities):,}")

MATCHING CHECK
Facilities without service records : 722
Service records without facilities : 2,841


## 7. Test Merge Strategies

Before creating the final master dataset, different merge strategies are tested to understand their impact on the available records.

The recommendation system requires both facility characteristics and service information. Therefore, comparing merge approaches helps determine the most appropriate dataset structure while minimizing unnecessary data loss.

The following merge approaches are evaluated:

- **Inner merge:** Retains only facilities with matching service records.
- **Left merge:** Retains all facilities and attaches available service information where available.
- **Outer merge:** Retains all records from both datasets for comparison and data auditing.

In [ ]:
# Test inner merge

inner_merge = facilities.merge(
    services,
    on="Facility ID",
    how="inner"
)

print("Inner Merge Shape:")
print(inner_merge.shape)

Inner Merge Shape:
(173721, 48)


In [ ]:
# Test left merge

left_merge = facilities.merge(
    services,
    on="Facility ID",
    how="left"
)

print("Left Merge Shape:")
print(left_merge.shape)

print("\nFacilities without services after left merge:")
print(left_merge["Service"].isna().sum())

Left Merge Shape:
(174443, 48)

Facilities without services after left merge:
722


In [ ]:
# Test outer merge

outer_merge = facilities.merge(
    services,
    on="Facility ID",
    how="outer",
    indicator=True
)

print("Outer Merge Shape:")
print(outer_merge.shape)

print("\nMerge Source Breakdown:")
print(outer_merge["_merge"].value_counts())

Outer Merge Shape:
(205800, 49)

Merge Source Breakdown:
_merge
both          173721
right_only     31357
left_only        722
Name: count, dtype: int64


## Merge Strategy Evaluation Summary

Three merge strategies were evaluated to determine the most appropriate approach for creating the recommendation dataset.

The inner merge produced **173,721 records**, representing facility-service combinations where matching Facility IDs existed in both datasets. The increase in rows reflects the one-to-many relationship between facilities and services.

The left merge produced **174,443 records**, retaining all facilities while attaching available service information. The additional 722 records represent facilities with no recorded services.

The outer merge identified **31,357 service records without matching facility profiles**, indicating service information linked to facilities that were not available in the cleaned facility dataset.

For the recommendation system, the preferred approach is to retain the facility dataset as the primary source and aggregate available service information at facility level. This preserves all registered facilities while allowing service-based recommendations.

The next step is therefore to create a facility-level master dataset by merging the datasets and transforming multiple service records into a consolidated service profile for each facility.

## 7. Merge Facilities and Services Datasets

The facilities dataset contains the core profile of each health facility, while the services dataset contains the healthcare services offered by each facility.

Since the recommendation system is designed to recommend health facilities, the facilities dataset is treated as the primary dataset. A left merge is therefore used to retain all registered facilities and attach available service information.

The merge is performed using `Facility ID` as the unique identifier.

The resulting dataset will contain facility attributes alongside their associated service records and will serve as the foundation for further transformation into a facility-level recommendation dataset.

In [ ]:
# Merge facilities with services

facility_services = facilities.merge(
    services,
    on="Facility ID",
    how="left",
    suffixes=("_facility", "_service")
)

# Check shape

print("Merged Dataset Shape:")
print(facility_services.shape)

display(facility_services.head())

Merged Dataset Shape:
(174443, 48)


,Facility ID,Facility Name_facility,Facility URL_facility,Facility Type,Keph Level,KHIS reporting,Open 24 hours,Published,Date established,Date requested,Date approved,Regulated,Regulation status,Regulating body,Registration number,License number,Category_facility,Owner,County,Sub County,Ward,Latitude,Longitude,Town,Description,Nearest landmark,Plot number,Total In-patient beds,General In-patient beds,Cots,Maternity beds,Emergency casualty beds,Intensive Care Unit beds,High Dependency Unit beds,Isolation beds,General theatres,Maternity theatres,Open weekends,NHIF accreditation,Open late night,Facility Name_service,Facility URL_service,Service,Category_service,Status,Original_Service,Standardized_Service,Service_Group
0,003564d4-7ad6-4a97-8323-8eda7b3475a8,Mata Arba Dispensary,https://kmhfr.health.go.ke/public/facilities/003564d4-7ad6-4a97-8323-8eda7b3475a8,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2014-10-06,2016-02-12,Yes,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Isiolo,Cherab,Cherab,1.093142,38.420842,-,-,-,-,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,Mata Arba Dispensary,https://kmhfr.health.go.ke/public/facilities/003564d4-7ad6-4a97-8323-8eda7b3475a8,Medical Outpatient Clinic,SPECIALIZED OUTPATIENTS CLINIC,Active,Medical Outpatient Clinic,Medical Outpatient Clinic,General Outpatient Services
1,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,TT toxoid for Pregnant Women,IMMUNISATION,Active,TT toxoid for Pregnant Women,TT Toxoid for Pregnant Women,Maternal Health Services
2,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Specialised ANC -high risk pregnancy,ANTENATAL CARE,Active,Specialised ANC -high risk pregnancy,Specialised Antenatal Care - High Risk Pregnancy,Maternal Health Services
3,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Short Acting Method,FAMILY PLANNING,Active,Short Acting Method,Short Acting Family Planning Method,Family Planning Services
4,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Screening using VIA/VILI,CANCER SCREENING,Active,Screening using VIA/VILI,Visual Inspection with Acetic Acid/Lugol's Iodine (VIA/VILI) Screening,Oncology Screening Services


In [ ]:
# Check merge completeness

print("Number of facilities in merged dataset:")
print(facility_services["Facility ID"].nunique())

print("\nMissing service records:")
print(facility_services["Service"].isna().sum())

Number of facilities in merged dataset:
12050

Missing service records:
722


# Create Master Facility List

The merged dataset (`facility_services`) currently contains information at the **facility-service level**, meaning each row represents a health facility and one of the services it provides.

Since a single health facility can offer multiple services, the same facility appears multiple times in the dataset.

For the recommendation system, we require a **facility-level dataset** where:

- One row represents one health facility.
- All services offered by the facility are consolidated into a single service profile.
- Facility characteristics such as location, ownership, and level of care are preserved.

This master facility list will serve as the foundation for:
- Service matching
- Facility ranking
- Distance calculations
- Recommendation generation

In [ ]:
# Inspect the merged dataset

print("Merged Dataset Shape:")
print(facility_services.shape)

print("\nNumber of Unique Facilities:")
print(facility_services["Facility ID"].nunique())

print("\nFirst Five Records:")
display(facility_services.head())

Merged Dataset Shape:
(174443, 48)

Number of Unique Facilities:
12050

First Five Records:


,Facility ID,Facility Name_facility,Facility URL_facility,Facility Type,Keph Level,KHIS reporting,Open 24 hours,Published,Date established,Date requested,Date approved,Regulated,Regulation status,Regulating body,Registration number,License number,Category_facility,Owner,County,Sub County,Ward,Latitude,Longitude,Town,Description,Nearest landmark,Plot number,Total In-patient beds,General In-patient beds,Cots,Maternity beds,Emergency casualty beds,Intensive Care Unit beds,High Dependency Unit beds,Isolation beds,General theatres,Maternity theatres,Open weekends,NHIF accreditation,Open late night,Facility Name_service,Facility URL_service,Service,Category_service,Status,Original_Service,Standardized_Service,Service_Group
0,003564d4-7ad6-4a97-8323-8eda7b3475a8,Mata Arba Dispensary,https://kmhfr.health.go.ke/public/facilities/003564d4-7ad6-4a97-8323-8eda7b3475a8,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2014-10-06,2016-02-12,Yes,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Isiolo,Cherab,Cherab,1.093142,38.420842,-,-,-,-,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,Mata Arba Dispensary,https://kmhfr.health.go.ke/public/facilities/003564d4-7ad6-4a97-8323-8eda7b3475a8,Medical Outpatient Clinic,SPECIALIZED OUTPATIENTS CLINIC,Active,Medical Outpatient Clinic,Medical Outpatient Clinic,General Outpatient Services
1,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,TT toxoid for Pregnant Women,IMMUNISATION,Active,TT toxoid for Pregnant Women,TT Toxoid for Pregnant Women,Maternal Health Services
2,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Specialised ANC -high risk pregnancy,ANTENATAL CARE,Active,Specialised ANC -high risk pregnancy,Specialised Antenatal Care - High Risk Pregnancy,Maternal Health Services
3,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Short Acting Method,FAMILY PLANNING,Active,Short Acting Method,Short Acting Family Planning Method,Family Planning Services
4,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Screening using VIA/VILI,CANCER SCREENING,Active,Screening using VIA/VILI,Visual Inspection with Acetic Acid/Lugol's Iodine (VIA/VILI) Screening,Oncology Screening Services


## Step 9.2 — Aggregate Services at Facility Level

The merged dataset contains multiple rows per facility because each facility can provide several healthcare services.

To create a facility-level dataset, services are grouped by `Facility ID` and combined into a single service profile.

Two new variables are created:

- **Services**: A list of all unique standardized services offered by the facility.
- **Service_Count**: The total number of unique services available at the facility.

This transformation creates a one-to-one relationship between `Facility ID` and service information.

In [ ]:
#Aggregate services by Facility ID

facility_services_summary = (
    facility_services
    .groupby("Facility ID")
    .agg(
        Services=(
            "Standardized_Service",
            lambda x: list(x.dropna().unique())
        ),
        Service_Count=(
            "Standardized_Service",
            "nunique"
        )
    )
    .reset_index()
)

# Check the result

print("Aggregated Services Dataset Shape:")
print(facility_services_summary.shape)

display(facility_services_summary.head())

Aggregated Services Dataset Shape:
(12050, 3)


,Facility ID,Services,Service_Count
0,0008896d-fc28-4059-b1dd-7b9dbe8b41f1,"[Tuberculosis Treatment, TT Toxoid for Pregnant Women, Short Acting Family Planning Method, Elimination of Mother to Child Transmission of HIV (eMTCT), Postnatal Care Services, General Outpatient Services, Long Acting Family Planning Method, Integrated Management of Newborn and Childhood Illnesses, Child Immunization Services, HIV Treatment and Care, HIV Counselling and Testing, Focused Antenatal Care, Condom Distribution and STI Prevention, Basic Emergency Obstetric Care (BEmOC)]",14
1,0011453e-2c33-4386-997e-b193754a9243,"[TT Toxoid for Pregnant Women, Specialised Antenatal Care - High Risk Pregnancy, Short Acting Family Planning Method, Visual Inspection with Acetic Acid/Lugol's Iodine (VIA/VILI) Screening, Elimination of Mother to Child Transmission of HIV (eMTCT), Pap Smear, General Outpatient Services, Natural Family Planning Method, Long Acting Family Planning Method, Child Immunization Services, Infection Prevention and Control (Occupational HIV Risk Mitigation), HIV Counselling and Testing, HIV Risk Reduction for Priority Populations and Geographies, Focused Antenatal Care, Condom Distribution and STI Prevention, Breast Cancer Screening]",16
2,0012e77d-f49b-4197-b4bb-bed6f21d4933,"[TT Toxoid for Pregnant Women, Short Acting Family Planning Method, Elimination of Mother to Child Transmission of HIV (eMTCT), General Outpatient Services, Newborn Care Services, Natural Family Planning Method, Long Acting Family Planning Method, Integrated Management of Newborn and Childhood Illnesses, Child Immunization Services, Hospital - Retail Services, HIV Treatment and Care, HIV Counselling and Testing, Focused Antenatal Care, Condom Distribution and STI Prevention, Class F, Class E, Basic Emergency Obstetric Care (BEmOC)]",17
3,001708fe-45a6-4fcc-9bb2-14a0b3a68bc7,[],0
4,001abd7e-e95d-46ed-91fd-175945ef2a0d,"[Basic Emergency Obstetric Care (BEmOC), Integrated Management of Acute Malnutrition, Growth Monitoring, Inpatient Services, General Outpatient Services, Facility Accredited by NHIF, Basic IMCI - Management of Acute Infections, Accident and Emergency Services]",8


In [ ]:
# Check that each facility appears only once

print("Unique Facility IDs:")
print(facility_services_summary["Facility ID"].nunique())

print("\nDuplicate Facility IDs:")
print(
    facility_services_summary["Facility ID"].duplicated().sum()
)

Unique Facility IDs:
12050

Duplicate Facility IDs:
0


## Step 9.3 — Create the Master Facility List

The aggregated service information is merged back with the original facilities dataset to create the final master facility list.

The facilities dataset is used as the primary table because it contains the complete registry of health facilities. A left merge is applied to ensure that all registered facilities are retained, including those without available service records.

The resulting dataset contains:

- One row per health facility
- Facility profile information
- Geographic information
- Facility level and ownership
- Consolidated service information
- Number of available services

This dataset will become the main input for the health facility recommendation system.

In [ ]:
# Create Master Facility List

master_facility_list = facilities.merge(
    facility_services_summary,
    on="Facility ID",
    how="left"
)

# Check shape

print("Master Facility List Shape:")
print(master_facility_list.shape)

display(master_facility_list.head())

Master Facility List Shape:
(12050, 42)


,Facility ID,Facility Name,Facility URL,Facility Type,Keph Level,KHIS reporting,Open 24 hours,Published,Date established,Date requested,Date approved,Regulated,Regulation status,Regulating body,Registration number,License number,Category,Owner,County,Sub County,Ward,Latitude,Longitude,Town,Description,Nearest landmark,Plot number,Total In-patient beds,General In-patient beds,Cots,Maternity beds,Emergency casualty beds,Intensive Care Unit beds,High Dependency Unit beds,Isolation beds,General theatres,Maternity theatres,Open weekends,NHIF accreditation,Open late night,Services,Service_Count
0,003564d4-7ad6-4a97-8323-8eda7b3475a8,Mata Arba Dispensary,https://kmhfr.health.go.ke/public/facilities/003564d4-7ad6-4a97-8323-8eda7b3475a8,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2014-10-06,2016-02-12,Yes,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Isiolo,Cherab,Cherab,1.093142,38.420842,-,-,-,-,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,[Medical Outpatient Clinic],1
1,0011453e-2c33-4386-997e-b193754a9243,Boka Dispensary,https://kmhfr.health.go.ke/public/facilities/0011453e-2c33-4386-997e-b193754a9243,Dispensary,Level 2,Yes,Yes,Yes,2016-03-17,2010-09-01,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Tana River,Bura,Bangale,-0.729680,38.996358,BANGALE TOWN,BOKA WELLS,BOKA PRMARY SCHOOL,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,"[TT Toxoid for Pregnant Women, Specialised Antenatal Care - High Risk Pregnancy, Short Acting Family Planning Method, Visual Inspection with Acetic Acid/Lugol's Iodine (VIA/VILI) Screening, Elimination of Mother to Child Transmission of HIV (eMTCT), Pap Smear, General Outpatient Services, Natural Family Planning Method, Long Acting Family Planning Method, Child Immunization Services, Infection Prevention and Control (Occupational HIV Risk Mitigation), HIV Counselling and Testing, HIV Risk Reduction for Priority Populations and Geographies, Focused Antenatal Care, Condom Distribution and STI Prevention, Breast Cancer Screening]",16
2,001ff20b-d75a-4377-a762-18b6d5345d30,Arito Langi Health Centre,https://kmhfr.health.go.ke/public/facilities/001ff20b-d75a-4377-a762-18b6d5345d30,Basic Health Centre,Level 3,Yes,Yes,Yes,2016-03-17,2009-10-31,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Kisumu,Seme,West Seme,-0.094480,34.449480,-,3 km from Kolenyo market along Kisumu Bondo road and is 11 kms from the district office,-,-,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,[],0
3,0012e77d-f49b-4197-b4bb-bed6f21d4933,Rhonda Health Centre,https://kmhfr.health.go.ke/public/facilities/0012e77d-f49b-4197-b4bb-bed6f21d4933,Comprehensive Health Centre,Level 3,Yes,Yes,Yes,2016-03-17,2014-01-30,2016-02-12,NaN,Pending Gazettement,-,-,-,Ministry of Health,Ministry of Health,Nakuru,Nakuru West,Rhoda,-0.289467,36.034856,-,Near Soko mjinga,-,nil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,"[TT Toxoid for Pregnant Women, Short Acting Family Planning Method, Elimination of Mother to Child Transmission of HIV (eMTCT), General Outpatient Services, Newborn Care Services, Natural Family Planning Method, Long Acting Family Planning Method, Integrated Management of Newborn and Childhood Illnesses, Child Immunization Services, Hospital - Retail Services, HIV Treatment and Care, HIV Counselling and Testing, Focused Antenatal Care, Condom Distribution and STI Prevention, Class F, Class E, Basic Emergency Obstetric Care (BEmOC)]",17
4,001e2b20-c18f-459b-b095-eb6e9d082db0,Maxcare Medical Clinic,https://kmhfr.health.go.ke/public/facilities/001e2b20-c18f-459b-b095-eb6e9d082db0,Medical Clinic,Level 2,Yes,Yes,Yes,2018-04-12,2019-09-24,2021-03-10,Yes,Licensed,-,-,-,Private Practice,Private Practice - Clinical Officer,Kiambu,Gatundu North,Githobokoni,-0.887340,36.804480,Gakoe Shopping Center,Near Gakoe Catholic Church,Gakoe catholic church,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Yes,NaN,NaN,"[Short Acting Family Planning Method, Postnatal 

In [ ]:
# Validate the master facility list

print("=" * 60)
print("MASTER FACILITY LIST VALIDATION")
print("=" * 60)

print(
    "Unique Facilities:",
    master_facility_list["Facility ID"].nunique()
)

print(
    "Duplicate Facility IDs:",
    master_facility_list["Facility ID"].duplicated().sum()
)

print(
    "Facilities without service records:",
    master_facility_list["Services"].isna().sum()
)

MASTER FACILITY LIST VALIDATION
Unique Facilities: 12050
Duplicate Facility IDs: 0
Facilities without service records: 0


In [ ]:
# Define output path

master_path = "/content/drive/MyDrive/Data/Afya guide/Clean_Data/master_facility_list.csv"

# Save master facility list

master_facility_list.to_csv(
    master_path,
    index=False
)

print("Master Facility List saved successfully!")
print(master_path)

Master Facility List saved successfully!
/content/drive/MyDrive/Data/Afya guide/Clean_Data/master_facility_list.csv


## Step 9.5 — Missing Value Assessment and Treatment

The Master Facility List combines facility attributes and service information.

Before developing the recommendation engine, missing values are reviewed and handled based on their meaning.

Different types of missing values require different approaches:

- Service information missing → interpreted as no recorded services.
- Geographic information missing → important because distance-based recommendations require coordinates.
- Facility characteristics missing → assessed depending on importance for ranking.
- Optional descriptive fields → retained as missing where appropriate.



In [ ]:
# Missing values summary

missing_summary = (
    master_facility_list.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]

,0
Open late night,9891
NHIF accreditation,9391
Open weekends,7616
Regulated,2745
KHIS reporting,1607
Open 24 hours,343
Plot number,341
Latitude,201
Date established,162
Published,160


## Step 9.5 — Missing Value Treatment

The Master Facility List contains missing values across different facility attributes.

Missing values are handled according to their role in the recommendation system:

- Service information: Missing values are treated as facilities without recorded services.
- Geographic information: Missing coordinates are flagged because they affect distance calculations.
- Capacity variables: Missing values are treated as zero where absence of reported capacity is meaningful.
- Categorical operational variables: Missing values are labelled as "Unknown" to distinguish unavailable information from negative values.
- Descriptive fields: Missing text values are replaced with "Not Available".



In [ ]:
# Handle missing service information

master_facility_list["Services"] = (
    master_facility_list["Services"]
    .apply(lambda x: [] if isinstance(x, float) else x)
)

# Replace missing service counts

master_facility_list["Service_Count"] = (
    master_facility_list["Service_Count"]
    .fillna(0)
    .astype(int)
)

print("Missing Services:")
print(master_facility_list["Services"].isna().sum())

print("\nMissing Service Count:")
print(master_facility_list["Service_Count"].isna().sum())

Missing Services:
0

Missing Service Count:
0


In [ ]:
# Create coordinate availability flag

master_facility_list["Has_Coordinates"] = (
    master_facility_list["Latitude"].notna()
    &
    master_facility_list["Longitude"].notna()
)


master_facility_list["Has_Coordinates"].value_counts()

,count
Has_Coordinates,
True,11849
False,201


In [ ]:
# Capacity variables

capacity_columns = [
    "Total In-patient beds",
    "General In-patient beds",
    "Cots",
    "Maternity beds",
    "Emergency casualty beds",
    "Intensive Care Unit beds",
    "High Dependency Unit beds",
    "Isolation beds",
    "General theatres",
    "Maternity theatres"
]


# Replace missing capacity values with zero

master_facility_list[capacity_columns] = (
    master_facility_list[capacity_columns]
    .fillna(0)
)

In [ ]:
categorical_columns = [
    "KHIS reporting",
    "Open 24 hours",
    "Published",
    "Regulated",
    "Regulation status",
    "Open weekends",
    "NHIF accreditation",
    "Open late night"
]


master_facility_list[categorical_columns] = (
    master_facility_list[categorical_columns]
    .fillna("Unknown")
)

In [ ]:
text_columns = [
    "Description",
    "Nearest landmark",
    "Town",
    "Plot number"
]


master_facility_list[text_columns] = (
    master_facility_list[text_columns]
    .fillna("Not Available")
)

In [ ]:
# Check remaining missing values

remaining_missing = (
    master_facility_list.isna()
    .sum()
    .sort_values(ascending=False)
)

remaining_missing[remaining_missing > 0]

,0
Latitude,201
Date established,162
Longitude,157
Date approved,141
Date requested,50


In [ ]:
# Define output path

clean_master_path = "/content/drive/MyDrive/Data/Afya guide/Clean_Data/master_facility_list_cleaned.csv"


# Save cleaned master facility list

master_facility_list.to_csv(
    clean_master_path,
    index=False
)


print("Cleaned Master Facility List saved successfully!")
print(clean_master_path)

Cleaned Master Facility List saved successfully!
/content/drive/MyDrive/Data/Afya guide/Clean_Data/master_facility_list_cleaned.csv


 Explore Existing Service Groups

The cleaned services dataset already contains standardized service names and higher-level service groups.Before creating the user-facing taxonomy, we first examine the available service groups to understand how healthcare services are organized within the KMHFR dataset.This helps identify the major categories of healthcare services that will be mapped to common user health needs.

In [ ]:
#Number of service groups
print("Number of Service Groups:")
print(services["Service_Group"].nunique())

#List all service groups
service_groups=(
    services["Service_Group"].
    value_counts().
    reset_index()
)
service_groups.column=["Service Group", "Number of Services"]
display(service_groups)

Number of Service Groups:
33


/tmp/ipykernel_656/470736183.py:11: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  service_groups.column=["Service Group", "Number of Services"]


,Service_Group,count
0,Family Planning Services,34190
1,Maternal Health Services,30860
2,HIV/AIDS Services,26615
3,Child Health Services,20407
4,General Outpatient Services,16134
5,Emergency Services,9687
6,Regulatory and Accreditation Services,8623
7,Pharmacy Services,6853
8,Oncology Screening Services,5460
9,Nutrition Services,4415


In [ ]:
#Inspect them by group
for group in sorted(services["Service_Group"].unique()):
    print("=" * 70)
    print(group.upper())

    group_services = (
        services.loc[
            services["Service_Group"] == group,
            "Standardized_Service"
        ]
        .drop_duplicates()
        .sort_values()
    )

    for service in group_services:
        print("-", service)

    print()

BLOOD TRANSFUSION SERVICES
- Blood Bank Services
- Blood Transfusion Services
- Regional Blood Bank
- Satellite Blood Transfusion Services

CANCER TREATMENT SERVICES
- Chemotherapy
- Cryotherapy
- Loop Electrosurgical Excision Procedure (LEEP)
- Radiotherapy

CARDIOLOGY SERVICES
- Echocardiography

CHILD HEALTH SERVICES
- Basic IMCI - Management of Acute Infections
- Child Immunization Services
- Comprehensive IMCI - Resuscitation Services
- Growth Monitoring
- HPV Vaccination
- Integrated Management of Newborn and Childhood Illnesses
- Paediatric Outpatient Clinic

CRITICAL CARE SERVICES
- High Dependency Services
- High Dependency Unit (HDU) Services
- ICU Services

DENTAL AND ORAL HEALTH SERVICES
- Basic Dental Services
- Dental Anaesthesiology Services
- Dental X-Ray
- Endodontist Services
- Oral and Maxillofacial Pathology Services
- Oral and Maxillofacial Radiology Services
- Oral and Maxillofacial Surgery Services
- Orthodontist Services
- Periodontist Services
- Prosthodontist 